> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **GPT Judge Evaluation**
This notebook measures *Confirmation Bias* using LLM-as-a-judge (with gpt-4o), instructing it to rate the coherence on a scale of 0-10.

In [1]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.gpt_judge import compute_gpt_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [ ]:
# Define the path to the generated results (JSONL format)
# file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
file_path = "../data/interim/5_mmlu_pro_gpt_4o_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

Loaded 2 samples.


,sample,model,claim,response_neutral,response_leading,response_contradictory
0,1,gpt-4o,How are nutrients and waste materials transpor...,The circulatory system plays a key role in tra...,The option A provided is incorrect and misrepr...,The claim presented as option A is inaccurate....


## **Evaluation**
Compute the evaluation metrics by sending the generated results to GPT, and extract the final score (parsed list).

In [3]:
# Compute the GPT metrics
df_evaluated = compute_gpt_metrics(df_results, model="gpt-4o", sleep_time=1.0)

Starting GPT evaluation as judge for 2 samples...
Row 1/2 evaluated.
Row 2/2 evaluated.


## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [4]:
# Aggregation of the average scores for each prompt type
mean_neutral = df_evaluated["score_neutral"].mean()
mean_leading = df_evaluated["score_leading"].mean()
mean_contra = df_evaluated["score_contradictory"].mean()

print("Results of the GPT Judge Evaluation:")
print(f"Average Support (Neutral Prompt):        {mean_neutral:.2f} / 10")
print(f"Average Support (Leading Prompt):        {mean_leading:.2f} / 10")
print(f"Average Support (Contradictory Prompt):  {mean_contra:.2f} / 10")

# If the Leading prompt increases adherence and the Contradictory prompt decreases it significantly, 
# the numerical discrepancy indicates that the tested model suffers from confirmation bias.

display(df_evaluated[["sample", "model", "claim", "score_neutral", "score_leading", "score_contradictory"]].head(10))

Results of the GPT Judge Evaluation:
Average Support (Neutral Prompt):        9.00 / 10
Average Support (Leading Prompt):        8.50 / 10
Average Support (Contradictory Prompt):  0.75 / 10


,sample,model,claim,score_neutral,score_leading,score_contradictory
0,1,gpt-4o,How are nutrients and waste materials transpor...,9.0,8.5,1.5
1,2,gpt-4o,"In comparison to Homo habilis, what gave Homo ...",9.0,8.5,0.0


## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [5]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_file = f"../data/interim/{base_name}_gpt.csv"

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")

Saved evaluation file to ../data/interim/5_mmlu_pro_gpt_4o_gpt.csv
